# AKNA B-cell noKO paper figure exploration

## Input/output configuration

Edit these paths before running the analysis. Relative paths are resolved from the repository root. Run the cells in order using the R kernel.

In [ ]:
params <- list(
  primary_workbook = "results/analysis/03_signal_comparison/reproducible_binding_sites_groups_noKO_signal_comparison.xlsx",
  metagene_csv = "results/external/metagene_profile/peaks_with_normalized_positions.csv",
  corrected_go_csv = "results/analysis/04_go/go_enrichment_all_WT_KO_enriched_corrected.csv",
  go_mapping_csv = "results/analysis/04_go/go_gene_to_GO_mapping_WT_KO_enriched_run.csv",
  annotation_gtf = "data/reference/gencode.vM25.annotation.gtf",
  string_summary_workbook = "results/external/STRING_analysis_resume.xlsx",
  tracks_dir = "results/preprocessing/akna/crosslinked_nucleotides",
  output_dir = "results/analysis/05_manuscript_figures"
)

project_root <- normalizePath(getwd(), mustWork = TRUE)
if (!file.exists(file.path(project_root, "workflow", "Snakefile"))) {
  project_root <- normalizePath(file.path(project_root, ".."), mustWork = TRUE)
}
if (!file.exists(file.path(project_root, "workflow", "Snakefile"))) {
  stop("Open this notebook from the repository root or its analysis directory.")
}
resolve_repo_path <- function(path) {
  if (grepl("^(/|[A-Za-z]:[/\\\\])", path)) path else file.path(project_root, path)
}
paths <- lapply(params, resolve_repo_path)


# Purpose

This notebook preserves the manuscript figure exploration performed for the
B-cell no-input-control AKNA iCLIP2 analysis. It prepares binding-site region
plots, corrected GO summaries, candidate loci for IGV, and target lists for
exploratory STRING network analysis. It does not run STRING itself.

In [ ]:

library(readxl)
library(dplyr)
library(tidyr)
library(ggplot2)
library(stringr)
library(forcats)
library(scales)
library(readr)
library(GO.db)
library(AnnotationDbi)
library(writexl)
library(topGO)
library(rtracklayer)
library(GenomicFeatures)

out_dir <- paths$output_dir
dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)
theme_set(theme_classic(base_size = 12))

save_plot <- function(plot, name, width = 6, height = 4) {
  ggsave(
    file.path(out_dir, paste0(name, ".pdf")), plot,
    width = width, height = height, useDingbats = FALSE
  )
  ggsave(
    file.path(out_dir, paste0(name, ".png")), plot,
    width = width, height = height, dpi = 300
  )
}

clean_list_columns <- function(data) {
  data[] <- lapply(data, function(column) {
    if (is.list(column)) {
      vapply(column, paste, collapse = ";", FUN.VALUE = character(1))
    } else {
      column
    }
  })
  data
}

write_workbook <- function(sheets, path) {
  sheets <- lapply(sheets, function(x) clean_list_columns(as.data.frame(x)))
  writexl::write_xlsx(sheets, path)
}


# Load primary data

In [ ]:
stopifnot(file.exists(paths$primary_workbook))
bs <- read_excel(
  paths$primary_workbook,
  sheet = "all_reproducible_binding_sites"
) %>%
  mutate(
    region = factor(region, levels = c("utr5", "cds", "utr3", "intron")),
    pass_threshold = as.logical(pass_threshold),
    WT_mean = as.numeric(WT_mean),
    KO_mean = as.numeric(KO_mean),
    fold_change = as.numeric(fold_change)
  )

metagene <- if (file.exists(paths$metagene_csv)) {
  read_csv(paths$metagene_csv, show_col_types = FALSE)
} else {
  warning("Optional metagene input not found: ", paths$metagene_csv)
  tibble()
}

message("Primary binding-site rows: ", nrow(bs))
message("Unique target genes: ", n_distinct(bs$gene_name))
message("Metagene rows: ", nrow(metagene))


## Threshold summary

In [ ]:
threshold_summary <- tibble(
  selection = c(
    "all reproducible binding sites",
    "fold_change > 1.5",
    "fold_change > 1.5 and WT_mean > 3",
    "fold_change > 1.5 and WT_mean > 6",
    "fold_change > 1.5 and WT_mean > 10"
  ),
  n_binding_sites = c(
    nrow(bs),
    sum(bs$pass_threshold, na.rm = TRUE),
    sum(bs$pass_threshold & bs$WT_mean > 3, na.rm = TRUE),
    sum(bs$pass_threshold & bs$WT_mean > 6, na.rm = TRUE),
    sum(bs$pass_threshold & bs$WT_mean > 10, na.rm = TRUE)
  ),
  n_genes = c(
    n_distinct(bs$gene_name),
    n_distinct(bs$gene_name[bs$pass_threshold]),
    n_distinct(bs$gene_name[bs$pass_threshold & bs$WT_mean > 3]),
    n_distinct(bs$gene_name[bs$pass_threshold & bs$WT_mean > 6]),
    n_distinct(bs$gene_name[bs$pass_threshold & bs$WT_mean > 10])
  )
)
write_csv(threshold_summary, file.path(out_dir, "threshold_summary.csv"))
threshold_summary


# Binding-site position

In [ ]:
format_region_summary <- function(data) {
  data %>%
    count(region, name = "n") %>%
    mutate(
      fraction = n / sum(n),
      region_label = str_replace_all(
        as.character(region),
        c("utr5" = "5' UTR", "utr3" = "3' UTR", "cds" = "CDS",
          "intron" = "Intron")
      ),
      label = paste0(
        region_label, "\n", n, " (", percent(fraction, accuracy = 0.1), ")"
      )
    )
}

region_summary_all <- format_region_summary(bs)
region_summary <- bs %>% filter(pass_threshold) %>% format_region_summary()
region_plot_source <- bs %>%
  filter(pass_threshold) %>%
  mutate(region = as.character(region))

write_csv(region_summary_all, file.path(out_dir, "binding_site_region_summary_all_reproducible.csv"))
write_csv(region_summary, file.path(out_dir, "binding_site_region_summary_WT_KO_enriched.csv"))
write_csv(region_plot_source, file.path(out_dir, "binding_site_region_plot_source_WT_KO_enriched.csv"))
write_workbook(
  list(
    region_summary_WT_KO_enriched = region_summary,
    region_summary_all_reproducible = region_summary_all,
    WT_KO_enriched_binding_sites = region_plot_source
  ),
  file.path(out_dir, "AKNA_noKO_binding_site_region_distribution_WT_KO_enriched.xlsx")
)


In [ ]:
region_palette <- c(
  "utr5" = "#6C8EBF", "cds" = "#82B366",
  "utr3" = "#F6C85F", "intron" = "#B85450"
)
region_summary_pie <- region_summary %>%
  arrange(region) %>%
  mutate(
    ymax = cumsum(fraction),
    ymin = lag(ymax, default = 0),
    label_pos = (ymin + ymax) / 2,
    tick_end = if_else(region %in% c("cds", "intron"), 1.05, 1.07),
    label_x = case_when(
      region == "cds" ~ 1.35,
      region == "intron" ~ 1.29,
      TRUE ~ 1.23
    )
  )

p_region_pie <- ggplot(region_summary_pie, aes(fill = region)) +
  geom_rect(
    aes(xmin = 0, xmax = 1, ymin = ymin, ymax = ymax),
    color = "white", linewidth = 0.5
  ) +
  geom_segment(
    aes(x = 1, xend = tick_end, y = label_pos, yend = label_pos),
    color = "grey35", linewidth = 0.35
  ) +
  geom_text(aes(x = label_x, y = label_pos, label = label), size = 3.2) +
  coord_polar(theta = "y") +
  xlim(0, 1.6) +
  scale_fill_manual(values = region_palette, drop = FALSE) +
  labs(title = "AKNA-enriched binding-site position") +
  theme_void(base_size = 12) +
  theme(plot.title = element_text(hjust = 0.5, face = "bold"),
        legend.position = "none")
p_region_pie
save_plot(p_region_pie, "figure_binding_site_region_pie", 5.5, 5.5)

p_region_bar <- region_summary %>%
  mutate(region_label = fct_recode(
    region, "5' UTR" = "utr5", "CDS" = "cds",
    "3' UTR" = "utr3", "Intron" = "intron"
  )) %>%
  ggplot(aes(x = region_label, y = n, fill = region)) +
  geom_col(width = 0.72) +
  geom_text(aes(label = n), vjust = -0.35) +
  scale_fill_manual(values = region_palette, guide = "none", drop = FALSE) +
  labs(x = NULL, y = "Binding sites",
       title = "AKNA-enriched binding sites by gene-body region") +
  expand_limits(y = max(region_summary$n) * 1.12)
save_plot(p_region_bar, "figure_binding_site_region_bar", 5, 3.5)


# Corrected GO enrichment figures

This section consumes the corrected output from analysis `04`; it does not
recalculate adjusted P values from rounded historical exports.

In [ ]:
stopifnot(file.exists(paths$corrected_go_csv))
go <- read_csv(paths$corrected_go_csv, show_col_types = FALSE)

go_theme_terms <- tribble(
  ~theme, ~GO.ID,
  "Translation / ribosome", "GO:0022627",
  "Translation / ribosome", "GO:0042274",
  "Translation / ribosome", "GO:0045727",
  "Translation / ribosome", "GO:0002183",
  "Translation / ribosome", "GO:0006446",
  "Translation / ribosome", "GO:0003743",
  "Translation / ribosome", "GO:0043022",
  "Translation / ribosome", "GO:0032040",
  "Nuclear / chromatin", "GO:0005654",
  "Nuclear / chromatin", "GO:0005730",
  "Nuclear / chromatin", "GO:0016607",
  "Nuclear / chromatin", "GO:0000785",
  "Nuclear / chromatin", "GO:0000122",
  "Nuclear / chromatin", "GO:0045944",
  "Nuclear / chromatin", "GO:0006606",
  "Nuclear / chromatin", "GO:0003682",
  "B-cell / immune function", "GO:0050853",
  "B-cell / immune function", "GO:0019886",
  "B-cell / immune function", "GO:0001782",
  "B-cell / immune function", "GO:0030183",
  "B-cell / immune function", "GO:0042611",
  "B-cell / immune function", "GO:0050869",
  "B-cell / immune function", "GO:0030888",
  "Cytoskeleton / microtubule", "GO:0072686",
  "Cytoskeleton / microtubule", "GO:0008608",
  "Cytoskeleton / microtubule", "GO:0005813",
  "Cytoskeleton / microtubule", "GO:0015629",
  "Cytoskeleton / microtubule", "GO:0034314",
  "Cytoskeleton / microtubule", "GO:0030027",
  "Cytoskeleton / microtubule", "GO:0051015",
  "Protein homeostasis / complexes", "GO:0050821",
  "Protein homeostasis / complexes", "GO:0031625",
  "Protein homeostasis / complexes", "GO:0044877",
  "Protein homeostasis / complexes", "GO:0032991",
  "Protein homeostasis / complexes", "GO:0032436",
  "Protein homeostasis / complexes", "GO:0051082",
  "Protein homeostasis / complexes", "GO:0140534"
)
generic_go_ids <- c(
  "GO:0003729", "GO:0019843", "GO:0003723",
  "GO:0003730", "GO:0005515", "GO:0042802"
)

go_functional <- go %>%
  filter(BH_adjusted_P <= 0.05, !GO.ID %in% generic_go_ids) %>%
  inner_join(go_theme_terms, by = "GO.ID") %>%
  filter(Expected_AKNA_target_genes > 0) %>%
  group_by(theme) %>%
  slice_min(weightFisher_P, n = 3, with_ties = FALSE) %>%
  ungroup() %>%
  mutate(
    term_label = str_wrap(Term, width = 46),
    term_label_export = str_replace_all(term_label, fixed("\n"), " ")
  )
write_csv(
  go_functional %>% dplyr::select(-term_label_export) %>%
    mutate(term_label = str_replace_all(term_label, fixed("\n"), " ")),
  file.path(out_dir, "go_functional_theme_terms_corrected.csv")
)


In [ ]:
p_go_functional <- go_functional %>%
  ggplot(aes(x = Fold_enrichment,
             y = fct_reorder(term_label, Fold_enrichment))) +
  geom_point(
    aes(size = Observed_AKNA_target_genes,
        color = minus_log10_adjusted_P),
    alpha = 0.9
  ) +
  facet_wrap(~theme, ncol = 1, scales = "free_y") +
  scale_color_gradient(low = "#DDEAF6", high = "#08306B") +
  labs(
    x = "Observed / expected AKNA target genes", y = NULL,
    title = "Functional themes enriched among AKNA target mRNAs"
  ) +
  theme_classic(base_family = "Helvetica", base_size = 7) +
  theme(axis.line.y = element_blank(), axis.ticks.y = element_blank())
save_plot(p_go_functional, "figure_go_functional_themes_corrected", 6.4, 6.6)

go_compact <- go_functional %>%
  filter(Fold_enrichment >= 3) %>%
  arrange(Fold_enrichment) %>%
  mutate(term_label = factor(str_wrap(Term, 52), levels = str_wrap(Term, 52)))
p_go_compact <- ggplot(
  go_compact,
  aes(x = Fold_enrichment, y = term_label)
) +
  geom_point(
    aes(size = Observed_AKNA_target_genes,
        color = minus_log10_adjusted_P),
    alpha = 0.9
  ) +
  scale_color_gradient(low = "#DDEAF6", high = "#08306B") +
  labs(
    x = "Observed / expected AKNA target genes", y = NULL,
    title = "GO terms enriched >=3-fold among AKNA target mRNAs"
  ) +
  theme_classic(base_family = "Helvetica", base_size = 7) +
  theme(axis.line.y = element_blank(), axis.ticks.y = element_blank())
save_plot(p_go_compact, "figure_go_compact_enrichment_ge3_corrected", 5.2, 3.8)


## AKNA-bound mRNAs in GO:0008608

This exploratory export uses the same GENCODE protein-coding background and
topGO graph propagation as analysis `04`.

In [ ]:
stopifnot(file.exists(paths$annotation_gtf), file.exists(paths$go_mapping_csv))
anno <- rtracklayer::import(paths$annotation_gtf, format = "GTF")
anno <- anno[anno$level != 3]
anno$transcript_support_level[is.na(anno$transcript_support_level)] <- 0
anno$transcript_support_level[anno$transcript_support_level == "NA"] <- 10
anno <- anno[anno$transcript_support_level %in% c(0, 1, 2, 3)]
anno_db <- makeTxDbFromGRanges(anno)
gns <- genes(anno_db)
idx <- match(gns$gene_id, anno$gene_id)
elementMetadata(gns) <- cbind(elementMetadata(gns), elementMetadata(anno)[idx, ])
background_genes <- unique(gns$gene_name[gns$gene_type == "protein_coding"])
background_genes <- background_genes[
  !is.na(background_genes) & background_genes != ""
]

go_mapping <- read_csv(paths$go_mapping_csv, show_col_types = FALSE)
gene_2_GO <- unstack(go_mapping[, c("go_id", "external_gene_name")])
candidate_list <- unique(bs$gene_name[bs$pass_threshold])
gene_list <- factor(as.integer(background_genes %in% candidate_list))
names(gene_list) <- background_genes
go_data_bp <- new(
  "topGOdata", ontology = "BP", allGenes = gene_list,
  annot = annFUN.gene2GO, gene2GO = gene_2_GO, nodeSize = 20
)
go_spindle_id <- "GO:0008608"
go_spindle_genes <- sort(intersect(
  genesInTerm(go_data_bp, go_spindle_id)[[1]], candidate_list
))
go_spindle_sites <- bs %>%
  filter(gene_name %in% go_spindle_genes) %>%
  mutate(
    GO.ID = go_spindle_id,
    GO_term = "attachment of spindle microtubules to kinetochore",
    binding_site_coordinate = paste0(seqnames, ":", start, "-", end),
    igv_locus_2kb = paste0(seqnames, ":", pmax(1, start - 1000), "-", end + 1000)
  ) %>%
  arrange(gene_name, desc(pass_threshold), desc(WT_mean), desc(fold_change))
go_spindle_summary <- go_spindle_sites %>%
  group_by(gene_name) %>%
  summarise(
    n_reproducible_binding_sites = n(),
    n_binding_sites_passing_WT_KO_threshold = sum(pass_threshold),
    max_WT_mean = max(WT_mean),
    max_fold_change = max(fold_change),
    .groups = "drop"
  )
write_csv(go_spindle_summary, file.path(out_dir, "GO0008608_bound_mRNAs_summary.csv"))
write_csv(go_spindle_sites, file.path(out_dir, "GO0008608_binding_sites.csv"))


# IGV candidate loci

In [ ]:
cytoskeleton_candidates <- c(
  "Iqgap1", "Rac1", "Rac2", "Pafah1b1", "Msn", "Tubb4b", "Tubb5",
  "Dynll2", "Mapre3"
)
immune_candidates <- c(
  "Cd79a", "Cd79b", "Ms4a1", "Cd19", "Cd74", "Ptprc", "H2-Aa",
  "H2-Ab1", "H2-K1", "H2-D1"
)
translation_candidates <- c(
  "Rps19", "Rps20", "Rps24", "Rpl4", "Rpl30", "Rpsa", "Eef1a1", "Eef2"
)

candidate_loci <- bs %>%
  filter(gene_name %in% c(
    cytoskeleton_candidates, immune_candidates, translation_candidates
  )) %>%
  mutate(
    candidate_group = case_when(
      gene_name %in% cytoskeleton_candidates ~ "cytoskeleton / microtubule-associated",
      gene_name %in% immune_candidates ~ "immune / B-cell biology",
      gene_name %in% translation_candidates ~ "translation / ribosomal"
    ),
    locus_start = pmax(1, start - 1000),
    locus_end = end + 1000,
    igv_locus = paste0(seqnames, ":", locus_start, "-", locus_end)
  ) %>%
  arrange(candidate_group, desc(pass_threshold), desc(WT_mean), KO_mean)
write_csv(candidate_loci, file.path(out_dir, "igv_candidate_loci.csv"))

recommended_loci <- candidate_loci %>%
  filter(gene_name %in% c(
    "Msn", "Rac1", "Rac2", "Tubb5", "Tubb4b", "Pafah1b1", "Cd79a", "Ms4a1"
  )) %>%
  group_by(gene_name) %>%
  arrange(desc(WT_mean), KO_mean, .by_group = TRUE) %>%
  slice_head(n = 2) %>%
  ungroup() %>%
  arrange(desc(pass_threshold), desc(WT_mean))
write_csv(recommended_loci, file.path(out_dir, "igv_recommended_loci.csv"))


In [ ]:
track_paths <- file.path(
  paths$tracks_dir,
  c(
    "WT1.plus.bw", "WT1.minus.bw", "WT2.plus.bw", "WT2.minus.bw",
    "WT3.plus.bw", "WT3.minus.bw", "KO1.plus.bw", "KO1.minus.bw",
    "KO2.plus.bw", "KO2.minus.bw"
  )
)
missing_tracks <- track_paths[!file.exists(track_paths)]
if (length(missing_tracks) > 0) {
  warning("Missing tracks: ", paste(missing_tracks, collapse = "; "))
}
session_lines <- c(
  "<?xml version=\"1.0\" encoding=\"UTF-8\" standalone=\"no\"?>",
  "<Session genome=\"mm10\" hasGeneTrack=\"true\" hasSequenceTrack=\"true\" version=\"8\">",
  "  <Resources>",
  paste0("    <Resource path=\"", track_paths[file.exists(track_paths)], "\"/>"),
  "  </Resources>",
  "</Session>"
)
writeLines(session_lines, file.path(out_dir, "akna_bcell_noKO_igv_tracks.xml"))


# Exploratory STRING target lists

In [ ]:
if (file.exists(paths$string_summary_workbook)) {
  string_summary <- read_excel(paths$string_summary_workbook, sheet = 1)
  write_csv(string_summary, file.path(out_dir, "string_analysis_resume.csv"))
  string_summary
} else {
  warning("Optional STRING summary not found: ", paths$string_summary_workbook)
}


In [ ]:
string_targets_fc15 <- bs %>%
  filter(pass_threshold) %>%
  distinct(gene_name) %>%
  arrange(gene_name)
string_targets_fc15_wt6 <- bs %>%
  filter(pass_threshold, WT_mean > 6) %>%
  distinct(gene_name) %>%
  arrange(gene_name)

write_csv(string_targets_fc15, file.path(out_dir, "string_targets_fc_gt_1p5.csv"))
write_csv(string_targets_fc15_wt6, file.path(out_dir, "string_targets_fc_gt_1p5_WTmean_gt_6.csv"))
writeLines(string_targets_fc15$gene_name, file.path(out_dir, "string_targets_fc_gt_1p5.txt"))
writeLines(string_targets_fc15_wt6$gene_name, file.path(out_dir, "string_targets_fc_gt_1p5_WTmean_gt_6.txt"))

top_targets <- bs %>%
  filter(pass_threshold) %>%
  group_by(gene_name) %>%
  summarise(
    n_sites = n(),
    max_WT_mean = max(WT_mean, na.rm = TRUE),
    min_KO_mean = min(KO_mean, na.rm = TRUE),
    max_fold_change = max(fold_change, na.rm = TRUE),
    regions = paste(sort(unique(as.character(region))), collapse = "; "),
    .groups = "drop"
  ) %>%
  arrange(desc(max_WT_mean), desc(max_fold_change))
write_csv(top_targets, file.path(out_dir, "top_targets_summary.csv"))


# Session information

In [ ]:
sessionInfo()
